<a href="https://colab.research.google.com/github/c5063565-coder/dnnls_final_project/blob/main/Experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
import torchvision.transforms.functional as FT
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import math
import os
import time
import json
import textwrap
import re

from bs4 import BeautifulSoup
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import BertTokenizer
from google.colab import drive
import gc

print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

Device: cuda


In [2]:
drive.mount('/content/gdrive')

BASE_FOLDER = '/content/gdrive/MyDrive/DNN/DL_Checkpoints'

FOLDERS = {
    'base':        BASE_FOLDER,
    'checkpoints': os.path.join(BASE_FOLDER, 'checkpoints'),
    'baseline':    os.path.join(BASE_FOLDER, 'experiments', 'baseline'),
    'exp1':        os.path.join(BASE_FOLDER, 'experiments', 'exp1_transformer_resnet'),
    'exp2':        os.path.join(BASE_FOLDER, 'experiments', 'exp2_contrastive'),
}

for name, path in FOLDERS.items():
    os.makedirs(path, exist_ok=True)
print("Drive folder structure:")
for name, path in FOLDERS.items():
    print(f"  {name:12} → {path}")
def save_checkpoint(model, optimizer, epoch, loss, filename):
    path = os.path.join(FOLDERS['checkpoints'], filename)
    torch.save({
        'epoch':                epoch,
        'model_state_dict':     model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict() if optimizer else None,
        'loss':                 loss,
    }, path)
    print(f"Checkpoint saved: checkpoints/{filename}")

def load_checkpoint(model, optimizer=None, filename="checkpoint.pth"):
    path = os.path.join(FOLDERS['checkpoints'], filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Not found: {path}")
    ckpt = torch.load(path, map_location=torch.device('cpu'))
    model.load_state_dict(ckpt['model_state_dict'])
    if optimizer and ckpt.get('optimizer_state_dict'):
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    print(f"Loaded: {filename}  (epoch {ckpt.get('epoch', 0)})")
    return model, optimizer, ckpt.get('epoch', 0), ckpt.get('loss', None)

def save_log(lines, experiment):
    filename = f"training_log_{experiment}.txt"
    path     = os.path.join(FOLDERS[experiment], filename)
    content  = '\n'.join(lines)

    with open(path, 'w') as f:       # save to Drive
        f.write(content)
    with open(filename, 'w') as f:   # save locally in Colab as backup
        f.write(content)

    print(f"Log saved → experiments/{experiment}/{filename}")

def save_figure(fig, filename, experiment):
    path = os.path.join(FOLDERS[experiment], filename)
    fig.savefig(path, dpi=150, bbox_inches='tight')
    print(f"Figure saved → experiments/{experiment}/{filename}")
print("\nAll utilities ready.")


Mounted at /content/gdrive
Drive folder structure:
  base         → /content/gdrive/MyDrive/DNN/DL_Checkpoints
  checkpoints  → /content/gdrive/MyDrive/DNN/DL_Checkpoints/checkpoints
  baseline     → /content/gdrive/MyDrive/DNN/DL_Checkpoints/experiments/baseline
  exp1         → /content/gdrive/MyDrive/DNN/DL_Checkpoints/experiments/exp1_transformer_resnet
  exp2         → /content/gdrive/MyDrive/DNN/DL_Checkpoints/experiments/exp2_contrastive

All utilities ready.


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.empty_cache()
gc.collect()

EMB_DIM= 16# size of token embeddings
LATENT_DIM= 16# size of latent space vectors
NUM_LAYERS= 1# number of LSTM layers (baseline)
DROPOUT= 0.1# dropout rate
N_EPOCHS= 15# training epochs for main model
BATCH_SIZE= 8# samples per training batch
LR= 0.001# learning rate
MAX_LEN= 120# max token length for text
IMAGE_HW= (60, 125)# height x width for all images

USE_TRANSFORMER_ENCODER= False# False = LSTM /True = Transformer
USE_RESNET_ENCODER= False# False = CNN /True = ResNet-18
USE_CONTRASTIVE_ROI= False# False = off/True = contrastive grounding
USE_FRAME_AWARE_GROUNDING= False# ties ROI to its specific frame text
USE_ENTITY_POOLING= False# entity consistency loss
USE_COT_TEXT= False# append CoT text to descriptions

#Contrastive loss weights
LAMBDA_CONTRAST= 0.10
LAMBDA_GROUND_MSE= 0.10
LAMBDA_REID= 0.10
LAMBDA_ENTITY_POOL= 0.05
CONTRASTIVE_TAU= 0.07#temperature for InfoNCE loss

EXP_NAME = "{}_{}_{}" .format(
    "transformer" if USE_TRANSFORMER_ENCODER else "lstm",
    "resnet"if USE_RESNET_ENCODER else "cnn",
    "contrast"if USE_CONTRASTIVE_ROI else "nocontrast"
)

if not USE_TRANSFORMER_ENCODER and not USE_RESNET_ENCODER:
    EXP_FOLDER = 'baseline'
elif USE_TRANSFORMER_ENCODER and USE_RESNET_ENCODER and not USE_CONTRASTIVE_ROI:
    EXP_FOLDER = 'exp1'
elif USE_TRANSFORMER_ENCODER and USE_RESNET_ENCODER and USE_CONTRASTIVE_ROI:
    EXP_FOLDER = 'exp2'
else:
    EXP_FOLDER = 'baseline' # fallback

print(f"Device:{device}")
print(f"Experiment:{EXP_NAME}")
print(f"Folder:experiments/{EXP_FOLDER}")
print(f"Transformer:{USE_TRANSFORMER_ENCODER}")
print(f"ResNet:{USE_RESNET_ENCODER}")
print(f"Contrastive:{USE_CONTRASTIVE_ROI}")

Device:cuda
Experiment:lstm_cnn_nocontrast
Folder:experiments/baseline
Transformer:False
ResNet:False
Contrastive:False


In [4]:
train_dataset = load_dataset("daniel3303/StoryReasoning", split="train")
test_dataset = load_dataset("daniel3303/StoryReasoning", split="test")

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"\nColumns in dataset: {train_dataset.column_names}")

sample = train_dataset[0]
print(f"\nSample keys: {list(sample.keys())}")
print(f"Number of frames: {sample['frame_count']}")
print(f"Number of images: {len(sample['images'])}")
print(f"Story text preview: {sample['story'][:200]}")
print(f"CoT preview: {str(sample.get('chain_of_thought', ''))[:200]}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/327M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/331M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/115M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3552 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/626 [00:00<?, ? examples/s]

Train samples: 3552
Test samples: 626

Columns in dataset: ['story_id', 'images', 'frame_count', 'chain_of_thought', 'story']

Sample keys: ['story_id', 'images', 'frame_count', 'chain_of_thought', 'story']
Number of frames: 17
Number of images: 17
Story text preview: <gdi image1>
In the sterile environment of a sparse room filled with <gdo obj1>cardboard boxes</gdo>, <gdo obj5>a blue blanket</gdo>, and <gdo bg26>a table</gdo>, <gdo char1>James</gdo> <gda char1>ent
CoT preview: ## Image 1

### Characters
| Character ID | Name | Description | Emotions | Actions | Narrative Function | Bounding Box |
|--------------|------|-------------|----------|---------|-------------------|


In [5]:
sample = train_dataset[0]
print(f"Story text preview: {sample['story'][:200]}")

Story text preview: <gdi image1>
In the sterile environment of a sparse room filled with <gdo obj1>cardboard boxes</gdo>, <gdo obj5>a blue blanket</gdo>, and <gdo bg26>a table</gdo>, <gdo char1>James</gdo> <gda char1>ent


In [6]:
def parse_gdi_text(text):
    soup = BeautifulSoup(text, 'html.parser')
    images = []
    for gdi in soup.find_all('gdi'):
        image_id = None
        if gdi.attrs:
            for attr_name in gdi.attrs:
                if 'image' in attr_name.lower():
                    image_id = attr_name.replace('image', '')
                    break
        if not image_id:
            match = re.search(r'<gdi\s+image(\d+)', str(gdi))
            if match:
                image_id = match.group(1)
        if not image_id:
            image_id = str(len(images) + 1)
        images.append({
            'image_id':image_id,
            'description': gdi.get_text().strip(),
            'objects':[o.get_text().strip() for o in gdi.find_all('gdo')],
            'actions':[a.get_text().strip() for a in gdi.find_all('gda')],
            'locations':[l.get_text().strip() for l in gdi.find_all('gdl')],
        })

    return images
def show_image(ax, image):
    ax.imshow(image.permute(1, 2, 0).cpu().numpy().clip(0, 1))

def _parse_markdown_table(block):
    lines = [l.rstrip() for l in block.splitlines()]
    table_lines = [l for l in lines if l.strip().startswith("|")]

    if len(table_lines) < 3:   # need at least header + separator + one row
        return []

    headers = [h.strip() for h in table_lines[0].strip("|").split("|")]
    rows = []

    for line in table_lines[2:]:   # skip header and separator rows
        cols = [c.strip() for c in line.strip("|").split("|")]
        if len(cols) == len(headers):
            rows.append(dict(zip(headers, cols)))
    return rows
def parse_cot_grounding(chain_of_thought):
    frames = {}
    pattern = re.compile(r"^##\s*Image\s+(\d+)", flags=re.MULTILINE)
    matches = list(pattern.finditer(chain_of_thought or ""))

    for i, m in enumerate(matches):
        idx = int(m.group(1)) - 1   # convert to 0-indexed
        start = m.end()
        end = matches[i+1].start() if i+1 < len(matches) else len(chain_of_thought)
        section = chain_of_thought[start:end]

        frames[idx] = {"characters": [], "objects": []}

        for tag, key in [("Characters", "characters"), ("Objects", "objects")]:
            hit = re.search(
                rf"###\s*{tag}(.*?)(?=\n###|\n##|$)",
                section, re.DOTALL
            )
            if hit:
                id_key = "Character ID" if key == "characters" else "Object ID"
                for row in _parse_markdown_table(hit.group(1)):
                    eid = row.get(id_key, "").strip()
                    bbox = row.get("Bounding Box", "").strip()
                    if eid and bbox:
                        try:
                            x1, y1, x2, y2 = [int(v) for v in bbox.split(",")]
                            frames[idx][key].append({
                                "id":   eid,
                                "bbox": [x1, y1, x2, y2]
                            })
                        except:
                            pass

    return frames
def crop_and_resize(pil_img, bbox, out_hw=IMAGE_HW):
    W, H = pil_img.size
    x1, y1, x2, y2 = bbox
    x1, x2 = max(0, x1), min(W-1, x2)
    y1, y2 = max(0, y1), min(H-1, y2)
    if x2 <= x1: x2 = min(W-1, x1+1)
    if y2 <= y1: y2 = min(H-1, y1+1)
    crop = pil_img.crop((x1, y1, x2, y2))
    return transforms.Compose([
        transforms.Resize(out_hw),
        transforms.ToTensor()
    ])(crop)
def pick_reid_pair(frames_cot):
    import random
    id_to_dets = {}
    for f_idx, content in frames_cot.items():
        for det in content.get("characters", []) + content.get("objects", []):
            eid = det.get("id")
            bbox = det.get("bbox")
            if eid and bbox:
                id_to_dets.setdefault(eid, []).append((f_idx, bbox))
    candidates = [e for e, d in id_to_dets.items() if len(d) >= 2]
    if not candidates:
        return None

    eid= random.choice(candidates)
    (f1, b1), (f2, b2) = random.sample(id_to_dets[eid], 2)
    return f1, f2, b1, b2, eid
def extract_cot_text_for_frame(cot, frame_idx, max_chars=600):
    if not cot:
        return ""
    pattern = re.compile(r"^##\s*Image\s+(\d+)", flags=re.MULTILINE)
    matches = list(pattern.finditer(cot))
    for i, m in enumerate(matches):
        if int(m.group(1)) - 1 == frame_idx:
            start  = m.end()
            end    = matches[i+1].start() if i+1 < len(matches) else len(cot)
            target = cot[start:end]
            lines = [
                l for l in target.splitlines()
                if not l.strip().startswith("|")
                and set(l.strip()) > set("-|:")
            ]
            text = " ".join(l.strip() for l in lines if l.strip())
            return re.sub(r"\s+", " ", text).strip()[:max_chars]

    return ""
sample      = train_dataset[0]
parsed      = parse_gdi_text(sample['story'])
cot_frames  = parse_cot_grounding(sample.get('chain_of_thought', ''))

print(f"Frames parsed from story : {len(parsed)}")
print(f"Frame 0 description: {parsed[0]['description'][:80]}")
print(f"Frame 0 objects: {parsed[0]['objects']}")
print(f"Frame 0 actions: {parsed[0]['actions']}")
print(f"Frame 0 locations: {parsed[0]['locations']}")
print(f"CoT frames with bboxes: {list(cot_frames.keys())}")
print("\nHelper functions ready.")


Frames parsed from story : 17
Frame 0 description: In the sterile environment of a sparse room filled with cardboard boxes, a blue 
Frame 0 objects: ['cardboard boxes', 'a blue blanket', 'a table', 'James', 'James', 'James']
Frame 0 actions: ['enters', 'enters', 'looks around']
Frame 0 locations: ['neutral lighting', 'simple, minimalistic architecture']
CoT frames with bboxes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]

Helper functions ready.
